# Module 02 — Supervised Learning
## 02-03: Decision Trees & Random Forests

**Objective:** Build decision tree splitting (Gini impurity, entropy, exhaustive search) and recursive tree construction from scratch, assemble a full sklearn-style classifier, then extend to a Random Forest via bagging and random feature subsampling — including out-of-bag error estimation and impurity-based feature importances.

**Prerequisites:** 1-07 (Probability & Statistics for ML), 1-08 (Information Theory for ML), 2-01 (Linear Regression), 2-02 (Logistic Regression & Binary Classification)

## Part 0 — Setup & Prerequisites

This notebook covers **decision trees** and **random forests**, two of the most interpretable and practically effective algorithms for tabular classification. We implement the full **CART** (Classification and Regression Trees) algorithm from scratch: computing impurity measures (Gini impurity and Shannon entropy), exhaustively searching every feature-threshold pair for the optimal split, and building a tree recursively until stopping conditions are met. Pre-pruning via `max_depth`, `min_samples_split`, and `min_samples_leaf` controls overfitting. We then build a **Random Forest** by combining **bagging** (training each tree on a bootstrap sample) with **random feature subsampling** (considering only $\sqrt{p}$ features per split), computing an out-of-bag (OOB) accuracy estimate as a free validation set.

**Prerequisites:** 1-07 (Probability & Statistics), 1-08 (Information Theory), 2-01, 2-02

In [ ]:
# ── Imports ──────────────────────────────────────────────────────────────────
import sys
import warnings
warnings.filterwarnings("ignore")

import random
import math
import time
from dataclasses import dataclass, field
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

from sklearn.datasets import load_iris, make_classification
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, f1_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay,
)
from sklearn.tree import DecisionTreeClassifier as SklearnDT
from sklearn.ensemble import RandomForestClassifier as SklearnRF
import sklearn

print(f"Python:       {sys.version.split()[0]}")
print(f"NumPy:        {np.__version__}")
print(f"Scikit-learn: {sklearn.__version__}")
print(f"PyTorch:      {torch.__version__}")
if torch.cuda.is_available():
    print(f"CUDA: {torch.version.cuda}")
    print(f"GPU:  {torch.cuda.get_device_name(0)}")

In [ ]:
# ── Reproducibility ───────────────────────────────────────────────────────────
SEED = 1103
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Classical ML notebook — no GPU needed
print(f"Random seed set: {SEED}")

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────
# Decision Tree hyperparameters
MAX_DEPTH: int = 5
MIN_SAMPLES_SPLIT: int = 2
MIN_SAMPLES_LEAF: int = 1

# Random Forest hyperparameters
N_ESTIMATORS: int = 100
RF_MAX_FEATURES: str = "sqrt"    # sqrt(n_features) features per split
RF_MAX_DEPTH: int | None = None  # Grow fully — bagging controls variance

# Dataset
TEST_SIZE: float = 0.20
N_SAMPLES_SYNTH: int = 1000
N_FEATURES_SYNTH: int = 10
N_CLASSES_SYNTH: int = 3

### Data Loading & EDA

We use two datasets:
- **Iris** (150 samples, 4 features, 3 classes): a clean benchmark well-suited for visualizing decision boundaries in 2D petal space.
- **Synthetic classification** (1,000 samples, 10 features, 3 classes): larger and noisier, designed to showcase the Random Forest's advantage over a single tree.

In [ ]:
# ── Iris Dataset ──────────────────────────────────────────────────────────────
iris = load_iris()
X_iris: np.ndarray = iris.data           # shape: (150, 4)
y_iris: np.ndarray = iris.target         # values: 0, 1, 2
feature_names_iris: list[str] = list(iris.feature_names)
class_names_iris: list[str] = list(iris.target_names)

X_iris_train, X_iris_test, y_iris_train, y_iris_test = train_test_split(
    X_iris, y_iris, test_size=TEST_SIZE, random_state=SEED, stratify=y_iris
)

print("Iris Dataset")
print(f"  Total samples : {len(X_iris)} | Features: {X_iris.shape[1]} | Classes: {len(class_names_iris)}")
print(f"  Train/Test    : {len(X_iris_train)} / {len(X_iris_test)}")
print(f"  Class dist (train): {dict(Counter(y_iris_train))}")
print(f"  Features      : {feature_names_iris}")

# ── Synthetic Classification Dataset ─────────────────────────────────────────
X_synth, y_synth = make_classification(
    n_samples=N_SAMPLES_SYNTH,
    n_features=N_FEATURES_SYNTH,
    n_informative=6,
    n_redundant=2,
    n_classes=N_CLASSES_SYNTH,
    random_state=SEED,
)

X_synth_train, X_synth_test, y_synth_train, y_synth_test = train_test_split(
    X_synth, y_synth, test_size=TEST_SIZE, random_state=SEED, stratify=y_synth
)

print("\nSynthetic Dataset")
print(f"  Total samples : {len(X_synth)} | Features: {X_synth.shape[1]} | Classes: {N_CLASSES_SYNTH}")
print(f"  Train/Test    : {len(X_synth_train)} / {len(X_synth_test)}")
print(f"  Class dist (train): {dict(Counter(y_synth_train))}")

# ── EDA: Iris feature distributions by class ─────────────────────────────────
colors = ["steelblue", "darkorange", "green"]
fig, axes = plt.subplots(2, 2, figsize=(10, 7))
axes = axes.flatten()
for feat_idx, ax in enumerate(axes):
    for cls_idx, cls_name in enumerate(class_names_iris):
        mask = y_iris == cls_idx
        ax.hist(X_iris[mask, feat_idx], bins=15, alpha=0.6,
                color=colors[cls_idx], label=cls_name, edgecolor="white")
    ax.set_xlabel(feature_names_iris[feat_idx], fontsize=9)
    ax.set_ylabel("Count")
    ax.set_title(feature_names_iris[feat_idx])
    ax.legend(fontsize=8)
plt.suptitle("Iris — Feature Distributions by Class", fontsize=13)
plt.tight_layout()
plt.show()

---
## Part 1 — Decision Trees from Scratch

A decision tree partitions the feature space by asking a sequence of yes/no questions: **"Is feature $j$ ≤ threshold $t$?"**. At each internal node, we choose the $(j, t)$ pair that most reduces impurity in the child nodes. Leaves store the majority class of the training samples that reached them.

### 1.1 Impurity Measures

We need a measure of how "mixed" the class labels are at a node. Two standard choices:

**Gini impurity** — probability of mislabeling a random sample:
$$\text{Gini}(y) = 1 - \sum_{k=1}^{K} p_k^2$$

**Shannon entropy** — expected bits of information needed to describe a label (see 1-08):
$$H(y) = -\sum_{k=1}^{K} p_k \log_2 p_k$$

Both equal 0 for a pure node ($p_k = 1$ for one class) and are maximized when classes are equally distributed. Gini is slightly faster to compute (no logarithm); entropy is the basis for *information gain*.

In [ ]:
def compute_gini(y: np.ndarray) -> float:
    """Compute the Gini impurity of a label array.

    Gini impurity is the probability that a randomly chosen sample would be
    incorrectly classified if labeled at random according to the class
    distribution at this node:  Gini(y) = 1 - sum_k p_k^2

    Args:
        y: 1-D integer array of class labels, shape (n_samples,).

    Returns:
        Gini impurity in [0, 1 - 1/K] for K classes. 0 means pure.
    """
    n = len(y)
    if n == 0:
        return 0.0
    counts = np.bincount(y)
    probs = counts / n
    return float(1.0 - np.sum(probs ** 2))


def compute_entropy(y: np.ndarray) -> float:
    """Compute the Shannon entropy of a label array (in bits).

    H(y) = -sum_k p_k * log2(p_k), where terms with p_k=0 are skipped.
    See 1-08 (Information Theory) for the derivation.

    Args:
        y: 1-D integer array of class labels, shape (n_samples,).

    Returns:
        Shannon entropy in bits. 0 means pure; max is log2(K) for K classes.
    """
    n = len(y)
    if n == 0:
        return 0.0
    counts = np.bincount(y)
    probs = counts[counts > 0] / n   # skip zero counts to avoid log(0)
    return float(-np.sum(probs * np.log2(probs)))


def compute_weighted_child_impurity(
    y_left: np.ndarray,
    y_right: np.ndarray,
    criterion: str = "gini",
) -> float:
    """Compute the weighted average impurity of two child nodes after a split.

    The weighted impurity is:
        Score = (n_left / n) * Impurity(left) + (n_right / n) * Impurity(right)

    Lower is better. The impurity *decrease* (gain) equals:
        parent_impurity - weighted_child_impurity

    Args:
        y_left: Labels in the left child (X[:, feat] <= threshold).
        y_right: Labels in the right child (X[:, feat] > threshold).
        criterion: Impurity measure to use — "gini" or "entropy".

    Returns:
        Weighted child impurity. Returns 0.0 if both children are empty.
    """
    n = len(y_left) + len(y_right)
    if n == 0:
        return 0.0
    impurity_fn = compute_gini if criterion == "gini" else compute_entropy
    weight_left = len(y_left) / n
    weight_right = len(y_right) / n
    return weight_left * impurity_fn(y_left) + weight_right * impurity_fn(y_right)


# ── Quick verification ────────────────────────────────────────────────────────
y_pure = np.array([1, 1, 1, 1])
y_mixed = np.array([0, 0, 1, 1])
y_very_mixed = np.array([0, 1, 2, 3])

print("Gini impurity:")
print(f"  Pure  (all same class): {compute_gini(y_pure):.4f}  (expected 0.0)")
print(f"  Mixed (50/50 binary)  : {compute_gini(y_mixed):.4f}  (expected 0.5)")
print(f"  Very mixed (4 classes): {compute_gini(y_very_mixed):.4f}  (expected 0.75)")
print("\nShannon entropy (bits):")
print(f"  Pure  (all same class): {compute_entropy(y_pure):.4f}  (expected 0.0)")
print(f"  Mixed (50/50 binary)  : {compute_entropy(y_mixed):.4f}  (expected 1.0 bit)")
print(f"  Very mixed (4 classes): {compute_entropy(y_very_mixed):.4f}  (expected 2.0 bits)")

In [ ]:
# ── Visualize impurity as a function of class probability (binary case) ───────
p_values = np.linspace(1e-9, 1 - 1e-9, 300)
gini_curve = [1 - p**2 - (1 - p)**2 for p in p_values]
entropy_curve = [-p * math.log2(p) - (1 - p) * math.log2(1 - p) for p in p_values]
# Scale entropy to [0, 0.5] range for visual comparison
entropy_curve_scaled = [e / 2.0 for e in entropy_curve]

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(p_values, gini_curve, lw=2, label="Gini impurity: $1 - p^2 - (1-p)^2$")
ax.plot(p_values, entropy_curve, lw=2, linestyle="--",
        label="Shannon entropy: $-p\\log_2 p - (1-p)\\log_2(1-p)$ (bits)")
ax.plot(p_values, entropy_curve_scaled, lw=1.5, linestyle=":",
        label="Entropy / 2 (scaled for comparison)")
ax.axvline(0.5, color="gray", linestyle=":", alpha=0.6, label="Maximum impurity ($p=0.5$)")
ax.set_xlabel("Probability of positive class $p$")
ax.set_ylabel("Impurity")
ax.set_title("Gini Impurity vs. Shannon Entropy (Binary Classification)")
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

print("Key insight: both measures agree on what is pure (0) and what is maximally mixed (0.5).")
print("Gini is faster (no log); entropy gives higher weight to confident wrong splits.")

### 1.2 Finding the Best Split

To find the best split at a node, we try every feature and every unique threshold value. For a feature with $m$ unique values, there are $m - 1$ candidate thresholds (the midpoints between consecutive sorted values). The split that most reduces the weighted child impurity is chosen.

With $n$ samples and $p$ features, the worst-case complexity per node is $O(np \log n)$ — $O(n \log n)$ to sort each of the $p$ features. The Random Forest adds **feature subsampling**: at each node, only $\sqrt{p}$ randomly chosen features are considered, reducing both cost and tree correlation.

In [ ]:
def find_best_split(
    X: np.ndarray,
    y: np.ndarray,
    max_features: int | None,
    criterion: str,
    rng: np.random.RandomState,
) -> tuple[int, float, float]:
    """Search all (feature, threshold) pairs for the split minimizing child impurity.

    Candidate thresholds for feature j are the midpoints between every pair of
    consecutive unique sorted values. The chosen split sends samples with
    X[:, feature_idx] <= threshold to the left child and the rest to the right.

    Args:
        X: Feature matrix at this node, shape (n_node_samples, n_features).
        y: Labels at this node, shape (n_node_samples,).
        max_features: Number of features to consider. None means use all.
            Set to int(sqrt(n_features)) for Random Forest.
        criterion: Impurity measure — "gini" or "entropy".
        rng: NumPy RandomState for reproducible feature subsampling.

    Returns:
        Tuple (best_feature_idx, best_threshold, best_score) where:
            best_feature_idx: Column index of the feature chosen for splitting.
            best_threshold:   Threshold value (go left if X[i, feat] <= threshold).
            best_score:       Weighted child impurity of the best split found.
    """
    n_samples, n_features = X.shape

    # Feature subsampling — core randomization in Random Forest
    if max_features is None or max_features >= n_features:
        candidate_features = np.arange(n_features)
    else:
        candidate_features = rng.choice(n_features, size=max_features, replace=False)

    best_score: float = float("inf")
    best_feature_idx: int = 0
    best_threshold: float = 0.0

    for feature_idx in candidate_features:
        column = X[:, feature_idx]
        sorted_unique = np.sort(np.unique(column))
        if len(sorted_unique) < 2:
            continue  # all values identical — no split possible
        # Midpoints between consecutive unique values as candidate thresholds
        thresholds = (sorted_unique[:-1] + sorted_unique[1:]) / 2.0

        for threshold in thresholds:
            left_mask = column <= threshold
            right_mask = ~left_mask
            # Skip degenerate (all-left or all-right) splits
            if left_mask.sum() == 0 or right_mask.sum() == 0:
                continue
            score = compute_weighted_child_impurity(
                y[left_mask], y[right_mask], criterion
            )
            if score < best_score:
                best_score = score
                best_feature_idx = int(feature_idx)
                best_threshold = float(threshold)

    return best_feature_idx, best_threshold, best_score


# ── Manual trace on a tiny dataset ───────────────────────────────────────────
X_trace = np.array([[1.0, 2.0], [1.5, 3.0], [4.0, 1.0], [4.5, 0.5]])
y_trace = np.array([0, 0, 1, 1])
rng_trace = np.random.RandomState(SEED)

feat, thr, score = find_best_split(X_trace, y_trace, max_features=None,
                                    criterion="gini", rng=rng_trace)
print(f"Best split on 4-sample toy data:")
print(f"  Feature index : {feat}  (feature 0 = x-axis)")
print(f"  Threshold     : {thr:.2f}")
print(f"  Child impurity: {score:.4f}  (should be 0.0 — perfectly separable)")
print(f"  Left (<=thr)  : {y_trace[X_trace[:, feat] <= thr]}")
print(f"  Right (>thr)  : {y_trace[X_trace[:, feat] > thr]}")

### 1.3 Building the Tree Recursively

The tree is built top-down using a simple recursive strategy:

1. Compute the node's impurity and majority class.
2. Check stopping conditions — if any apply, make this node a **leaf**:
   - Reached `max_depth`
   - Fewer than `min_samples_split` samples
   - Node is already pure (impurity == 0)
   - No split reduces impurity
3. Otherwise, find the best split, partition samples, and recurse into left and right children.

Each `Node` stores the split parameters (for internal nodes) or the predicted class (for leaves), plus metadata used for feature importances.

In [ ]:
@dataclass
class Node:
    """A single node in a decision tree.

    Internal nodes store the split condition (feature_idx, threshold) and
    pointers to left/right children. Leaf nodes store a predicted class label.

    Attributes:
        feature_idx:  Feature column used for splitting (None for leaves).
        threshold:    Split value; go left if X[i, feature_idx] <= threshold.
        left:         Left child node (None for leaves).
        right:        Right child node (None for leaves).
        value:        Majority-class prediction (set for leaf nodes only).
        impurity:     Gini/entropy before splitting at this node.
        n_samples:    Number of training samples reaching this node.
        class_counts: Per-class sample counts at this node.
        depth:        Depth of this node in the tree (root = 0).
    """
    feature_idx: int | None = None
    threshold: float | None = None
    left: "Node | None" = field(default=None, repr=False)
    right: "Node | None" = field(default=None, repr=False)
    value: int | None = None
    impurity: float = 0.0
    n_samples: int = 0
    class_counts: "np.ndarray | None" = field(default=None, repr=False)
    depth: int = 0

    @property
    def is_leaf(self) -> bool:
        """Return True when this node is a leaf (no further splitting)."""
        return self.value is not None


def build_tree(
    X: np.ndarray,
    y: np.ndarray,
    depth: int,
    max_depth: int | None,
    min_samples_split: int,
    min_samples_leaf: int,
    max_features: int | None,
    criterion: str,
    n_classes: int,
    rng: np.random.RandomState,
) -> Node:
    """Recursively build a decision tree, returning the root Node.

    Args:
        X: Feature matrix at this node, shape (n_node_samples, n_features).
        y: Labels at this node, shape (n_node_samples,).
        depth: Current depth in the tree (root starts at 0).
        max_depth: Maximum allowed depth. None = grow until pure or data runs out.
        min_samples_split: Minimum samples required to consider splitting.
        min_samples_leaf: Minimum samples required in each child node.
        max_features: Features to consider per split. None = all features.
        criterion: Impurity measure — "gini" or "entropy".
        n_classes: Total number of classes (for consistent class_counts shape).
        rng: NumPy RandomState for reproducibility.

    Returns:
        A Node object representing this subtree.
    """
    impurity_fn = compute_gini if criterion == "gini" else compute_entropy
    n_samples = len(y)
    class_counts = np.bincount(y, minlength=n_classes)
    node_impurity = impurity_fn(y)
    majority_class = int(np.argmax(class_counts))

    node = Node(
        impurity=node_impurity,
        n_samples=n_samples,
        class_counts=class_counts,
        depth=depth,
    )

    # ── Pre-pruning stopping conditions ───────────────────────────────────────
    if (
        (max_depth is not None and depth >= max_depth)
        or n_samples < min_samples_split
        or node_impurity == 0.0
    ):
        node.value = majority_class
        return node

    # ── Find best split ───────────────────────────────────────────────────────
    feature_idx, threshold, split_score = find_best_split(
        X, y, max_features=max_features, criterion=criterion, rng=rng
    )

    # No improvement — make a leaf
    if split_score >= node_impurity:
        node.value = majority_class
        return node

    # ── Apply split ───────────────────────────────────────────────────────────
    left_mask = X[:, feature_idx] <= threshold
    right_mask = ~left_mask

    # Enforce min_samples_leaf in both children
    if left_mask.sum() < min_samples_leaf or right_mask.sum() < min_samples_leaf:
        node.value = majority_class
        return node

    node.feature_idx = feature_idx
    node.threshold = threshold
    node.left = build_tree(
        X[left_mask], y[left_mask],
        depth=depth + 1,
        max_depth=max_depth,
        min_samples_split=min_samples_split,
        min_samples_leaf=min_samples_leaf,
        max_features=max_features,
        criterion=criterion,
        n_classes=n_classes,
        rng=rng,
    )
    node.right = build_tree(
        X[right_mask], y[right_mask],
        depth=depth + 1,
        max_depth=max_depth,
        min_samples_split=min_samples_split,
        min_samples_leaf=min_samples_leaf,
        max_features=max_features,
        criterion=criterion,
        n_classes=n_classes,
        rng=rng,
    )
    return node


def predict_sample(node: Node, x: np.ndarray) -> int:
    """Traverse the decision tree to predict the class for one sample.

    Args:
        node: Current (or root) Node of the decision tree.
        x: Feature vector of shape (n_features,).

    Returns:
        Predicted class label as an integer.
    """
    if node.is_leaf:
        return node.value  # type: ignore[return-value]
    if x[node.feature_idx] <= node.threshold:  # type: ignore[index]
        return predict_sample(node.left, x)    # type: ignore[arg-type]
    return predict_sample(node.right, x)        # type: ignore[arg-type]

---
## Part 2 — Putting It All Together

We now wrap the building blocks into a complete `DecisionTreeClassifier` with a scikit-learn-compatible API (`fit`, `predict`, `score`). The class also computes **impurity-based feature importances**: each internal node's contribution is weighted by the fraction of training samples that passed through it, scaled by the impurity decrease it achieves.

$$\text{importance}(j) = \sum_{\text{nodes where feature}=j} \frac{n_{\text{node}}}{n_{\text{total}}} \left(\text{Gini}_{\text{node}} - \frac{n_L}{n_{\text{node}}} \text{Gini}_L - \frac{n_R}{n_{\text{node}}} \text{Gini}_R\right)$$

These raw importance values are then normalized to sum to 1.

In [ ]:
class DecisionTreeClassifier:
    """Decision tree classifier built from scratch using recursive CART splitting.

    Implements Gini impurity or Shannon entropy as the splitting criterion with
    pre-pruning controls (max_depth, min_samples_split, min_samples_leaf).
    Supports random feature subsampling for use as a base estimator in
    RandomForestClassifier.

    Attributes:
        max_depth:            Maximum tree depth. None = unlimited.
        min_samples_split:    Minimum samples to attempt a split.
        min_samples_leaf:     Minimum samples required in each child.
        max_features:         Features considered per split. None = all.
        criterion:            Splitting criterion — "gini" or "entropy".
        random_state:         Seed for reproducible feature subsampling.
        root_:                Root Node after fitting.
        n_classes_:           Number of unique classes.
        n_features_:          Number of input features.
        feature_importances_: Normalized impurity-decrease importances.
        classes_:             Sorted array of unique class labels.
    """

    def __init__(
        self,
        max_depth: int | None = None,
        min_samples_split: int = 2,
        min_samples_leaf: int = 1,
        max_features: int | None = None,
        criterion: str = "gini",
        random_state: int = SEED,
    ) -> None:
        """Initialize DecisionTreeClassifier with pruning and splitting hyperparameters.

        Args:
            max_depth: Maximum depth. None = grow until all leaves are pure.
            min_samples_split: Minimum samples to attempt a split.
            min_samples_leaf: Minimum samples required in each child leaf.
            max_features: Features considered per split. None = all features.
            criterion: Splitting criterion -- "gini" or "entropy".
            random_state: Integer seed for reproducible feature subsampling.
        """
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.min_samples_leaf = min_samples_leaf
        self.max_features = max_features
        self.criterion = criterion
        self.random_state = random_state
        # Set after fit()
        self.root_: Node | None = None
        self.n_classes_: int = 0
        self.n_features_: int = 0
        self.feature_importances_: np.ndarray | None = None
        self.classes_: np.ndarray | None = None

    def fit(self, X: np.ndarray, y: np.ndarray) -> "DecisionTreeClassifier":
        """Fit the decision tree to training data.

        Args:
            X: Feature matrix, shape (n_samples, n_features).
            y: Integer class labels, shape (n_samples,).

        Returns:
            self — the fitted classifier.
        """
        self.classes_ = np.unique(y)
        self.n_classes_ = len(self.classes_)
        self.n_features_ = X.shape[1]
        rng = np.random.RandomState(self.random_state)

        self.root_ = build_tree(
            X, y,
            depth=0,
            max_depth=self.max_depth,
            min_samples_split=self.min_samples_split,
            min_samples_leaf=self.min_samples_leaf,
            max_features=self.max_features,
            criterion=self.criterion,
            n_classes=self.n_classes_,
            rng=rng,
        )
        self.feature_importances_ = self._compute_feature_importances()
        return self

    def predict(self, X: np.ndarray) -> np.ndarray:
        """Predict class labels for each row in X.

        Args:
            X: Feature matrix, shape (n_samples, n_features).

        Returns:
            Predicted class labels, shape (n_samples,).
        """
        if self.root_ is None:
            raise RuntimeError("Call fit() before predict().")
        return np.array([predict_sample(self.root_, x) for x in X])

    def score(self, X: np.ndarray, y: np.ndarray) -> float:
        """Compute mean accuracy on the given data.

        Args:
            X: Feature matrix, shape (n_samples, n_features).
            y: True labels, shape (n_samples,).

        Returns:
            Accuracy in [0, 1].
        """
        return float(np.mean(self.predict(X) == y))

    def get_depth(self) -> int:
        """Return the maximum depth of the fitted tree.

        Returns:
            Maximum depth (0 means a single leaf node).
        """
        def _depth(node: Node | None) -> int:
            """Recursively compute max depth."""
            if node is None or node.is_leaf:
                return 0
            return 1 + max(_depth(node.left), _depth(node.right))
        return _depth(self.root_)

    def count_leaves(self) -> int:
        """Count the total number of leaf nodes.

        Returns:
            Number of leaf nodes.
        """
        def _count(node: Node | None) -> int:
            """Recursively count leaves."""
            if node is None:
                return 0
            if node.is_leaf:
                return 1
            return _count(node.left) + _count(node.right)
        return _count(self.root_)

    def _compute_feature_importances(self) -> np.ndarray:
        """Compute normalized impurity-decrease feature importances.

        Each internal node contributes:
            (n_node / n_total) * (parent_impurity - weighted_child_impurity)
        summed over all nodes where a given feature was used.

        Returns:
            Importance array of shape (n_features,) that sums to 1.0.
        """
        importances = np.zeros(self.n_features_)
        total_samples = self.root_.n_samples if self.root_ else 1

        def _traverse(node: Node) -> None:
            """Accumulate importance contribution from an internal node."""
            if node.is_leaf or node.feature_idx is None:
                return
            n = node.n_samples
            n_left = node.left.n_samples if node.left else 0
            n_right = node.right.n_samples if node.right else 0
            left_imp = node.left.impurity if node.left else 0.0
            right_imp = node.right.impurity if node.right else 0.0
            weighted_child = (n_left / n) * left_imp + (n_right / n) * right_imp
            impurity_decrease = (n / total_samples) * (node.impurity - weighted_child)
            importances[node.feature_idx] += impurity_decrease
            _traverse(node.left)   # type: ignore[arg-type]
            _traverse(node.right)  # type: ignore[arg-type]

        _traverse(self.root_)  # type: ignore[arg-type]
        total = importances.sum()
        if total > 0:
            importances /= total
        return importances


    def predict_proba(self, X: np.ndarray) -> np.ndarray:
        """Predict class probabilities using training-sample fractions at each leaf.

        Each leaf node stores the fraction of training samples from each class
        that reached it. These fractions are the predicted probabilities.

        Args:
            X: Feature matrix, shape (n_samples, n_features).

        Returns:
            Probability matrix, shape (n_samples, n_classes).
        """
        if self.root_ is None:
            raise RuntimeError("Call fit() before predict_proba().")

        def _leaf_proba(node: Node, x: np.ndarray) -> np.ndarray:
            """Return class probability vector at the leaf node reached by x."""
            if node.is_leaf:
                total = node.class_counts.sum() if node.class_counts is not None else 1
                if total == 0 or node.class_counts is None:
                    proba = np.ones(self.n_classes_) / self.n_classes_
                else:
                    proba = node.class_counts / total
                return proba
            if x[node.feature_idx] <= node.threshold:  # type: ignore[index]
                return _leaf_proba(node.left, x)        # type: ignore[arg-type]
            return _leaf_proba(node.right, x)           # type: ignore[arg-type]

        return np.vstack([_leaf_proba(self.root_, x) for x in X])
    def __repr__(self) -> str:
        return (
            f"DecisionTreeClassifier(criterion='{self.criterion}', "
            f"max_depth={self.max_depth}, "
            f"min_samples_split={self.min_samples_split})"
        )

#### Sanity Check — Toy Data

Before training on real data, we verify correctness on examples where the answer is known:
- A **perfectly separable** dataset — the tree should achieve 100% accuracy.
- An **XOR** pattern — requires depth ≥ 2 to solve.

In [ ]:
# ── Sanity check 1: linearly separable ────────────────────────────────────────
X_easy = np.array([[1.0], [2.0], [3.0], [10.0], [11.0], [12.0]])
y_easy = np.array([0, 0, 0, 1, 1, 1])

dt_easy = DecisionTreeClassifier(max_depth=1, random_state=SEED)
dt_easy.fit(X_easy, y_easy)

print("Sanity Check 1 — Linearly Separable:")
print(f"  Threshold learned: {dt_easy.root_.threshold:.2f}")
print(f"  Accuracy:          {dt_easy.score(X_easy, y_easy):.0%}   (expected 100%)")
print(f"  Threshold in (3, 10)? {3.0 < dt_easy.root_.threshold < 10.0}")

# ── Sanity check 2: XOR — needs depth >= 2 ────────────────────────────────────
X_xor = np.array([[0.0, 0.0], [0.0, 1.0], [1.0, 0.0], [1.0, 1.0]])
y_xor = np.array([0, 1, 1, 0])   # XOR: same class iff both features identical

dt_depth1 = DecisionTreeClassifier(max_depth=1, random_state=SEED)
dt_depth3 = DecisionTreeClassifier(max_depth=3, random_state=SEED)
dt_depth1.fit(X_xor, y_xor)
dt_depth3.fit(X_xor, y_xor)

print("\nSanity Check 2 — XOR pattern:")
print(f"  Depth-1 stump accuracy:   {dt_depth1.score(X_xor, y_xor):.0%}  (expected 50% — XOR not linearly separable)")
print(f"  Depth-3 tree  accuracy:   {dt_depth3.score(X_xor, y_xor):.0%}  (expected 100%)")
print(f"  Depth-3 actual depth:     {dt_depth3.get_depth()}")
print(f"  Depth-3 leaf count:       {dt_depth3.count_leaves()}")

### 2.2 Random Forest — Bagging & Feature Subsampling

A single decision tree grown to full depth perfectly memorizes training data (variance is high). A **Random Forest** reduces variance by averaging predictions from many decorrelated trees.

Two sources of randomness de-correlate the trees:

1. **Bagging (Bootstrap Aggregating):** Each tree is trained on a *bootstrap sample* — $n$ samples drawn *with replacement* from the training set. About $1 - (1 - 1/n)^n \approx 63.2\%$ of unique samples appear in each bootstrap; the remaining $\approx 36.8\%$ are **out-of-bag (OOB)** samples, giving us a free validation set.

2. **Random feature subsampling:** At each split, only $\lfloor\sqrt{p}\rfloor$ of the $p$ features are considered. This ensures trees disagree even on the same training data, preventing any single strong feature from dominating all trees.

In [ ]:
def bootstrap_sample(
    X: np.ndarray,
    y: np.ndarray,
    rng: np.random.RandomState,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Draw a bootstrap sample (sampling with replacement) from (X, y).

    A bootstrap of size n samples n indices uniformly with replacement.
    On average, ~63.2% of unique indices appear in the bootstrap, and
    the remaining ~36.8% are out-of-bag (OOB) — useful for free validation.

    Args:
        X: Feature matrix, shape (n_samples, n_features).
        y: Labels, shape (n_samples,).
        rng: NumPy RandomState for reproducibility.

    Returns:
        Tuple (X_boot, y_boot, oob_indices) where:
            X_boot:      Bootstrapped features, shape (n_samples, n_features).
            y_boot:      Bootstrapped labels, shape (n_samples,).
            oob_indices: Original indices NOT drawn (out-of-bag samples).
    """
    n_samples = len(y)
    boot_indices = rng.randint(0, n_samples, size=n_samples)
    oob_indices = np.setdiff1d(np.arange(n_samples), np.unique(boot_indices))
    return X[boot_indices], y[boot_indices], oob_indices


# ── Verify the 63.2% unique-sample theorem ────────────────────────────────────
n_demo = 10_000
demo_rng = np.random.RandomState(SEED)
boot_idx = demo_rng.randint(0, n_demo, size=n_demo)
unique_in_bag = len(np.unique(boot_idx))
print(f"Bootstrap Illustration (n={n_demo:,} samples)")
print(f"  Unique in-bag  : {unique_in_bag:,} ({unique_in_bag/n_demo:.2%})")
print(f"  OOB samples    : {n_demo - unique_in_bag:,} ({(n_demo-unique_in_bag)/n_demo:.2%})")
print(f"  Theory (1-1/e) : {1 - 1/math.e:.4f}  ({(1 - 1/math.e):.2%})")


class RandomForestClassifier:
    """Random Forest built from scratch using bagging and random feature subsets.

    Each of the n_estimators trees is trained on an independent bootstrap
    sample of the training data. At each split only max_features (typically
    sqrt(p)) features are considered, decorrelating the trees and reducing
    variance. Prediction is by majority vote. OOB accuracy is estimated during
    training as a free approximation of hold-out accuracy.

    Attributes:
        n_estimators:         Number of trees.
        max_depth:            Max depth of each tree. None = unlimited.
        min_samples_split:    Min samples to split a node.
        min_samples_leaf:     Min samples per leaf.
        max_features:         Features per split — int, "sqrt", or "log2".
        criterion:            Impurity measure — "gini" or "entropy".
        random_state:         Master seed for the ensemble.
        trees_:               List of fitted DecisionTreeClassifier objects.
        oob_score_:           Out-of-bag accuracy estimate.
        feature_importances_: Mean impurity-decrease averaged over all trees.
        n_classes_:           Number of unique classes.
        n_features_:          Number of input features.
    """

    def __init__(
        self,
        n_estimators: int = 100,
        max_depth: int | None = None,
        min_samples_split: int = 2,
        min_samples_leaf: int = 1,
        max_features: int | str = "sqrt",
        criterion: str = "gini",
        random_state: int = SEED,
    ) -> None:
        """Initialize RandomForestClassifier with ensemble and tree hyperparameters.

        Args:
            n_estimators: Number of trees in the forest.
            max_depth: Maximum depth of each tree. None = unlimited.
            min_samples_split: Minimum samples to split a node.
            min_samples_leaf: Minimum samples required in each leaf.
            max_features: Features per split -- int, "sqrt", or "log2".
            criterion: Impurity measure -- "gini" or "entropy".
            random_state: Master seed for the ensemble.
        """
        self.n_estimators = n_estimators
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.min_samples_leaf = min_samples_leaf
        self.max_features = max_features
        self.criterion = criterion
        self.random_state = random_state
        self.trees_: list[DecisionTreeClassifier] = []
        self.oob_score_: float = 0.0
        self.feature_importances_: np.ndarray | None = None
        self.n_classes_: int = 0
        self.n_features_: int = 0

    def _resolve_max_features(self, n_features: int) -> int:
        """Convert max_features to a concrete integer count.

        Args:
            n_features: Total number of features in the dataset.

        Returns:
            Integer number of features to consider at each split.
        """
        if isinstance(self.max_features, int):
            return self.max_features
        if self.max_features == "sqrt":
            return max(1, int(math.sqrt(n_features)))
        if self.max_features == "log2":
            return max(1, int(math.log2(n_features)))
        return n_features  # None or unrecognized — use all

    def fit(self, X: np.ndarray, y: np.ndarray) -> "RandomForestClassifier":
        """Fit the random forest on training data.

        Each tree is trained on a bootstrap sample. OOB votes are accumulated
        across trees to estimate generalization accuracy without a held-out set.

        Args:
            X: Feature matrix, shape (n_samples, n_features).
            y: Integer class labels, shape (n_samples,).

        Returns:
            self — the fitted random forest.
        """
        n_samples, n_features = X.shape
        self.n_classes_ = len(np.unique(y))
        self.n_features_ = n_features
        max_feat_int = self._resolve_max_features(n_features)
        master_rng = np.random.RandomState(self.random_state)

        oob_votes = np.zeros((n_samples, self.n_classes_), dtype=int)
        self.trees_ = []

        for _ in range(self.n_estimators):
            tree_rng = np.random.RandomState(master_rng.randint(0, 2 ** 31))
            X_boot, y_boot, oob_idx = bootstrap_sample(X, y, rng=tree_rng)

            tree = DecisionTreeClassifier(
                max_depth=self.max_depth,
                min_samples_split=self.min_samples_split,
                min_samples_leaf=self.min_samples_leaf,
                max_features=max_feat_int,
                criterion=self.criterion,
                random_state=tree_rng.randint(0, 2 ** 31),
            )
            tree.fit(X_boot, y_boot)
            self.trees_.append(tree)

            # Accumulate OOB votes from this tree
            if len(oob_idx) > 0:
                oob_preds = tree.predict(X[oob_idx])
                for sample_i, pred in zip(oob_idx, oob_preds):
                    oob_votes[sample_i, pred] += 1

        # Compute OOB accuracy (only for samples seen out-of-bag at least once)
        has_oob = oob_votes.sum(axis=1) > 0
        if has_oob.sum() > 0:
            oob_predictions = np.argmax(oob_votes[has_oob], axis=1)
            self.oob_score_ = float(np.mean(oob_predictions == y[has_oob]))

        # Average feature importances over all trees
        importances_list = [
            t.feature_importances_
            for t in self.trees_
            if t.feature_importances_ is not None
        ]
        if importances_list:
            self.feature_importances_ = np.mean(importances_list, axis=0)
        return self

    def predict_proba(self, X: np.ndarray) -> np.ndarray:
        """Predict class probabilities as fraction of vote from each tree.

        Args:
            X: Feature matrix, shape (n_samples, n_features).

        Returns:
            Probability matrix, shape (n_samples, n_classes).
        """
        votes = np.zeros((len(X), self.n_classes_), dtype=int)
        for tree in self.trees_:
            preds = tree.predict(X)
            for i, pred in enumerate(preds):
                votes[i, pred] += 1
        return votes / self.n_estimators

    def predict(self, X: np.ndarray) -> np.ndarray:
        """Predict class labels by majority vote across all trees.

        Args:
            X: Feature matrix, shape (n_samples, n_features).

        Returns:
            Predicted class labels, shape (n_samples,).
        """
        return np.argmax(self.predict_proba(X), axis=1)

    def score(self, X: np.ndarray, y: np.ndarray) -> float:
        """Compute mean accuracy on the given data.

        Args:
            X: Feature matrix, shape (n_samples, n_features).
            y: True labels, shape (n_samples,).

        Returns:
            Accuracy in [0, 1].
        """
        return float(np.mean(self.predict(X) == y))

    def __repr__(self) -> str:
        return (
            f"RandomForestClassifier("
            f"n_estimators={self.n_estimators}, "
            f"max_features='{self.max_features}', "
            f"max_depth={self.max_depth})"
        )

In [ ]:
# ── RF sanity check — known-separable clusters ────────────────────────────────
X_rf_toy = np.vstack([
    np.random.RandomState(SEED).randn(20, 2) + np.array([0, 0]),
    np.random.RandomState(SEED + 1).randn(20, 2) + np.array([5, 5]),
])
y_rf_toy = np.array([0] * 20 + [1] * 20)

rf_toy = RandomForestClassifier(n_estimators=20, max_depth=3, random_state=SEED)
rf_toy.fit(X_rf_toy, y_rf_toy)

print("RF Sanity Check — Two Gaussian Clusters:")
print(f"  Accuracy : {rf_toy.score(X_rf_toy, y_rf_toy):.0%}   (expected ~100%)")
print(f"  OOB Score: {rf_toy.oob_score_:.2%}")
print(f"  Trees built: {len(rf_toy.trees_)}")

# Verify OOB score is close to train accuracy for clean data
print(f"  OOB and train accuracy within 10%: {abs(rf_toy.oob_score_ - rf_toy.score(X_rf_toy, y_rf_toy)) < 0.10}")

---
## Part 3 — Training on Real Datasets

We train on two datasets:
1. **Decision Tree on Iris** — well-suited for 2D decision boundary visualization.
2. **Random Forest on Synthetic** — 10 features, noisy classes; shows the ensemble advantage.

For each, we compare our from-scratch implementation against scikit-learn's optimized versions to validate correctness.

In [ ]:
# ── Decision Tree on Iris ─────────────────────────────────────────────────────
start = time.time()
dt_iris = DecisionTreeClassifier(
    max_depth=MAX_DEPTH,
    min_samples_split=MIN_SAMPLES_SPLIT,
    min_samples_leaf=MIN_SAMPLES_LEAF,
    criterion="gini",
    random_state=SEED,
)
dt_iris.fit(X_iris_train, y_iris_train)
dt_iris_time = time.time() - start

train_acc_dt_iris = dt_iris.score(X_iris_train, y_iris_train)
test_acc_dt_iris  = dt_iris.score(X_iris_test,  y_iris_test)

print("From-Scratch Decision Tree — Iris")
print(f"  Train accuracy : {train_acc_dt_iris:.2%}")
print(f"  Test  accuracy : {test_acc_dt_iris:.2%}")
print(f"  Tree depth     : {dt_iris.get_depth()} (max allowed: {MAX_DEPTH})")
print(f"  Leaf nodes     : {dt_iris.count_leaves()}")
print(f"  Fit time       : {dt_iris_time:.4f}s")

### Library Comparison — Decision Tree

sklearn's `DecisionTreeClassifier` uses the same CART algorithm but is optimized in Cython. Accuracy should be nearly identical; the gap (if any) reflects minor implementation differences in how ties are broken between equally good splits.

In [ ]:
# ── Sklearn Decision Tree — Iris ─────────────────────────────────────────────
start = time.time()
sklearn_dt_iris = SklearnDT(
    max_depth=MAX_DEPTH,
    min_samples_split=MIN_SAMPLES_SPLIT,
    min_samples_leaf=MIN_SAMPLES_LEAF,
    criterion="gini",
    random_state=SEED,
)
sklearn_dt_iris.fit(X_iris_train, y_iris_train)
sklearn_dt_iris_time = time.time() - start

sklearn_train_acc = sklearn_dt_iris.score(X_iris_train, y_iris_train)
sklearn_test_acc  = sklearn_dt_iris.score(X_iris_test,  y_iris_test)

print("Sklearn Decision Tree — Iris")
print(f"  Train accuracy : {sklearn_train_acc:.2%}")
print(f"  Test  accuracy : {sklearn_test_acc:.2%}")
print(f"  Tree depth     : {sklearn_dt_iris.get_depth()}")
print(f"  Leaf nodes     : {sklearn_dt_iris.get_n_leaves()}")
print(f"  Fit time       : {sklearn_dt_iris_time:.4f}s")

acc_delta = abs(test_acc_dt_iris - sklearn_test_acc)
print(f"\nAccuracy delta (from-scratch vs sklearn): {acc_delta:.4f}")
status = "MATCH" if acc_delta < 0.05 else "CHECK — notable difference"
print(f"Status: {status}")

# Compare feature importances
print("\nFeature importances comparison (Iris):")
fi_scratch = np.round(dt_iris.feature_importances_, 3)
fi_sklearn  = np.round(sklearn_dt_iris.feature_importances_, 3)
for name, s, sk in zip(feature_names_iris, fi_scratch, fi_sklearn):
    print(f"  {name:30s}  scratch: {s:.3f}  sklearn: {sk:.3f}")

In [ ]:
# ── Random Forest on Synthetic Dataset ───────────────────────────────────────
print("Training Random Forest (from scratch) — this may take 30–60 seconds...")
start = time.time()
rf_synth = RandomForestClassifier(
    n_estimators=N_ESTIMATORS,
    max_depth=RF_MAX_DEPTH,
    min_samples_split=MIN_SAMPLES_SPLIT,
    min_samples_leaf=MIN_SAMPLES_LEAF,
    max_features=RF_MAX_FEATURES,
    criterion="gini",
    random_state=SEED,
)
rf_synth.fit(X_synth_train, y_synth_train)
rf_train_time = time.time() - start

rf_train_acc = rf_synth.score(X_synth_train, y_synth_train)
rf_test_acc  = rf_synth.score(X_synth_test,  y_synth_test)

print(f"From-Scratch Random Forest — Synthetic Dataset ({rf_train_time:.1f}s)")
print(f"  Train accuracy : {rf_train_acc:.2%}")
print(f"  Test  accuracy : {rf_test_acc:.2%}")
print(f"  OOB  accuracy  : {rf_synth.oob_score_:.2%}   (free validation estimate)")
print(f"  Trees built    : {len(rf_synth.trees_)}")

In [ ]:
# ── Sklearn Random Forest — Synthetic ────────────────────────────────────────
start = time.time()
sklearn_rf_synth = SklearnRF(
    n_estimators=N_ESTIMATORS,
    max_depth=RF_MAX_DEPTH,
    min_samples_split=MIN_SAMPLES_SPLIT,
    min_samples_leaf=MIN_SAMPLES_LEAF,
    max_features=RF_MAX_FEATURES,
    criterion="gini",
    oob_score=True,
    random_state=SEED,
    n_jobs=1,
)
sklearn_rf_synth.fit(X_synth_train, y_synth_train)
sklearn_rf_time = time.time() - start

sklearn_rf_test_acc = sklearn_rf_synth.score(X_synth_test, y_synth_test)
print("Sklearn Random Forest — Synthetic Dataset")
print(f"  Test  accuracy : {sklearn_rf_test_acc:.2%}")
print(f"  OOB   accuracy : {sklearn_rf_synth.oob_score_:.2%}")
print(f"  Fit time       : {sklearn_rf_time:.1f}s  (Cython-optimized)")
print(f"\nAccuracy gap (scratch vs sklearn): {abs(rf_test_acc - sklearn_rf_test_acc):.4f}")

---
## Part 4 — Evaluation & Analysis

We evaluate with accuracy and macro-F1, produce a summary comparison table, visualize decision boundaries in 2D feature space, plot feature importances, run ablation studies on depth and tree count, and examine confident misclassifications.

In [ ]:
# ── Results Summary Table ─────────────────────────────────────────────────────
# Also train a single DT on synthetic for direct comparison with RF
dt_synth = DecisionTreeClassifier(max_depth=MAX_DEPTH, random_state=SEED)
dt_synth.fit(X_synth_train, y_synth_train)
dt_synth_acc = dt_synth.score(X_synth_test, y_synth_test)

# Majority-class baseline
majority_class = Counter(y_synth_train).most_common(1)[0][0]
baseline_preds = np.full(len(y_synth_test), majority_class)
baseline_acc = accuracy_score(y_synth_test, baseline_preds)
baseline_f1  = f1_score(y_synth_test, baseline_preds, average="macro", zero_division=0)

results_data: dict[str, list] = {
    "Model": [
        "Majority-Class Baseline",
        "DT (scratch) — Iris",
        "DT (sklearn) — Iris",
        "DT (scratch) — Synth",
        "RF (scratch) — Synth",
        "RF (sklearn) — Synth",
    ],
    "Test Accuracy": [
        baseline_acc,
        test_acc_dt_iris,
        sklearn_test_acc,
        dt_synth_acc,
        rf_test_acc,
        sklearn_rf_test_acc,
    ],
    "F1 Macro": [
        baseline_f1,
        f1_score(y_iris_test,  dt_iris.predict(X_iris_test),           average="macro"),
        f1_score(y_iris_test,  sklearn_dt_iris.predict(X_iris_test),   average="macro"),
        f1_score(y_synth_test, dt_synth.predict(X_synth_test),         average="macro"),
        f1_score(y_synth_test, rf_synth.predict(X_synth_test),         average="macro"),
        f1_score(y_synth_test, sklearn_rf_synth.predict(X_synth_test), average="macro"),
    ],
}

results_df = pd.DataFrame(results_data)
results_df["Test Accuracy"] = results_df["Test Accuracy"].map("{:.2%}".format)
results_df["F1 Macro"]      = results_df["F1 Macro"].map("{:.4f}".format)
print(results_df.to_string(index=False))

### Decision Boundary Visualization

We project Iris onto its two most discriminative features (petal length and petal width) to produce interpretable 2D decision boundaries. The decision tree's boundary consists of axis-aligned rectangles (each split is horizontal or vertical), while the Random Forest smooths these into softer regions via majority vote.

In [ ]:
def plot_decision_boundary(
    model: DecisionTreeClassifier | RandomForestClassifier,
    X: np.ndarray,
    y: np.ndarray,
    feature_names: list[str],
    class_names: list[str],
    title: str = "Decision Boundary",
    h: float = 0.02,
) -> None:
    """Plot 2D decision boundary for a two-feature classifier.

    Generates a dense meshgrid, predicts class for every grid point, and
    fills regions with class colors. Actual training samples are overlaid
    as scatter points with class-specific colors.

    Args:
        model: Fitted classifier with a predict() method.
        X: Two-column feature matrix, shape (n_samples, 2).
        y: True labels, shape (n_samples,).
        feature_names: Names of the two features (for axis labels).
        class_names: Names of each class (for the legend).
        title: Plot title.
        h: Meshgrid step size (smaller = finer resolution, slower).
    """
    marker_colors = ["steelblue", "darkorange", "green", "red"]
    x_min, x_max = X[:, 0].min() - 0.4, X[:, 0].max() + 0.4
    y_min, y_max = X[:, 1].min() - 0.4, X[:, 1].max() + 0.4
    xx, yy = np.meshgrid(
        np.arange(x_min, x_max, h),
        np.arange(y_min, y_max, h),
    )
    grid_preds = model.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

    fig, ax = plt.subplots(figsize=(7, 5))
    cmap_bg = plt.cm.get_cmap("tab10", len(class_names))
    ax.contourf(xx, yy, grid_preds, alpha=0.25, cmap=cmap_bg)
    ax.contour(xx, yy, grid_preds, colors="gray", linewidths=0.5, alpha=0.5)
    for cls_idx, cls_name in enumerate(class_names):
        mask = y == cls_idx
        ax.scatter(
            X[mask, 0], X[mask, 1],
            color=marker_colors[cls_idx % len(marker_colors)],
            edgecolors="k", linewidths=0.5, s=50, label=cls_name, zorder=3,
        )
    ax.set_xlabel(feature_names[0])
    ax.set_ylabel(feature_names[1])
    ax.set_title(title)
    ax.legend(loc="upper left", fontsize=9)
    plt.tight_layout()
    plt.show()


# ── Use petal features (indices 2, 3) for 2D visualization ───────────────────
petal_feat_idx = [2, 3]
petal_names = [feature_names_iris[i] for i in petal_feat_idx]
X_iris_2d = X_iris[:, petal_feat_idx]

X_2d_train, X_2d_test, y_2d_train, y_2d_test = train_test_split(
    X_iris_2d, y_iris, test_size=TEST_SIZE, random_state=SEED, stratify=y_iris
)

dt_2d = DecisionTreeClassifier(max_depth=4, random_state=SEED)
rf_2d = RandomForestClassifier(n_estimators=50, max_depth=5, random_state=SEED)
dt_2d.fit(X_2d_train, y_2d_train)
rf_2d.fit(X_2d_train, y_2d_train)

plot_decision_boundary(
    dt_2d, X_iris_2d, y_iris,
    feature_names=petal_names,
    class_names=class_names_iris,
    title=f"Decision Tree (depth={dt_2d.get_depth()}) — Iris Petal Features",
)
plot_decision_boundary(
    rf_2d, X_iris_2d, y_iris,
    feature_names=petal_names,
    class_names=class_names_iris,
    title="Random Forest (50 trees) — Iris Petal Features",
)
print(f"DT  2D test acc : {dt_2d.score(X_2d_test, y_2d_test):.2%}")
print(f"RF  2D test acc : {rf_2d.score(X_2d_test, y_2d_test):.2%}")
print("Note: RF boundary is smoother — no single axis-aligned boundary dominates.")

### Feature Importances

Impurity-based feature importances measure how much each feature reduces total impurity across all splits, weighted by the fraction of samples passing through each node. A high importance means the feature consistently produces large, high-coverage impurity reductions.

For Random Forest, importances are averaged over all trees — this smooths out the noise of any single tree's choices and gives a more reliable importance ranking than a single decision tree.

In [ ]:
# ── Feature Importance Comparison ─────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# ── Left: Iris DT importances ────────────────────────────────────────────────
dt_fi = dt_iris.feature_importances_
sk_fi = sklearn_dt_iris.feature_importances_
x_pos = np.arange(len(dt_fi))
width = 0.35

axes[0].bar(x_pos - width / 2, dt_fi, width, label="Scratch", color="steelblue",   alpha=0.85)
axes[0].bar(x_pos + width / 2, sk_fi, width, label="Sklearn", color="darkorange", alpha=0.85)
axes[0].set_xticks(x_pos)
axes[0].set_xticklabels(
    [n.replace(" (cm)", "") for n in feature_names_iris], rotation=20, ha="right", fontsize=8
)
axes[0].set_ylabel("Normalized importance")
axes[0].set_title("DT Feature Importances — Iris\n(scratch vs sklearn)")
axes[0].legend()

# ── Right: Synthetic RF importances (top features) ───────────────────────────
rf_fi = rf_synth.feature_importances_
top_k = 8
sorted_rf_idx = np.argsort(rf_fi)[::-1][:top_k]
feat_labels = [f"Feature {i}" for i in range(len(rf_fi))]

axes[1].bar(
    range(top_k),
    rf_fi[sorted_rf_idx],
    color="green", alpha=0.8, edgecolor="k",
)
axes[1].set_xticks(range(top_k))
axes[1].set_xticklabels(
    [feat_labels[i] for i in sorted_rf_idx], rotation=15, ha="right"
)
axes[1].set_ylabel("Mean impurity decrease")
axes[1].set_title("RF Feature Importances — Synthetic\n(top 8 features, averaged over 100 trees)")

plt.suptitle("Impurity-Based Feature Importances", fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

print("DT Scratch importances:", np.round(dt_iris.feature_importances_, 3))
print("DT Sklearn importances:", np.round(sklearn_dt_iris.feature_importances_, 3))
print("\nInsight: petal length and petal width dominate — they separate Iris classes cleanly.")

### Ablation Study — Depth & Number of Trees

**Decision tree depth** controls the bias-variance tradeoff directly: a depth-1 stump underfits (high bias), while an unlimited tree overfits (high variance on training data). The gap between train and test accuracy widens as depth increases — a clear overfitting signal.

**Random Forest tree count** shows a different pattern: adding more trees always helps or at worst stays flat (averaging reduces variance monotonically). The OOB accuracy closely tracks test accuracy without any held-out data.

In [ ]:
# ── Ablation 1: max_depth vs. accuracy on Synthetic Dataset ──────────────────
depths = list(range(1, 16))
train_accs_ablation: list[float] = []
test_accs_ablation:  list[float] = []

for d in depths:
    dt_abl = DecisionTreeClassifier(max_depth=d, random_state=SEED)
    dt_abl.fit(X_synth_train, y_synth_train)
    train_accs_ablation.append(dt_abl.score(X_synth_train, y_synth_train))
    test_accs_ablation.append(dt_abl.score(X_synth_test, y_synth_test))

best_depth_idx = int(np.argmax(test_accs_ablation))
best_depth     = depths[best_depth_idx]
best_test_acc  = test_accs_ablation[best_depth_idx]

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(depths, train_accs_ablation, "o-", label="Train accuracy", color="steelblue")
ax.plot(depths, test_accs_ablation,  "s--", label="Test accuracy",  color="darkorange")
ax.axvline(best_depth, color="gray", linestyle=":", alpha=0.7,
           label=f"Best depth = {best_depth}")
ax.set_xlabel("max_depth")
ax.set_ylabel("Accuracy")
ax.set_title("Decision Tree: Depth vs. Accuracy — Overfitting Analysis")
ax.legend()
plt.tight_layout()
plt.show()

print(f"Depth  1 (stump)  : train={train_accs_ablation[0]:.2%}, test={test_accs_ablation[0]:.2%}  (underfitting)")
print(f"Depth  {best_depth} (optimal): train={train_accs_ablation[best_depth_idx]:.2%}, test={best_test_acc:.2%}")
print(f"Depth 15 (deep)   : train={train_accs_ablation[-1]:.2%}, test={test_accs_ablation[-1]:.2%}  (overfitting)")

In [ ]:
# ── Ablation 2: n_estimators vs. accuracy (RF) ───────────────────────────────
estimator_counts = [1, 5, 10, 20, 50, 100]
rf_test_accs_abl: list[float] = []
rf_oob_accs_abl:  list[float] = []

for n_est in estimator_counts:
    rf_abl = RandomForestClassifier(
        n_estimators=n_est,
        max_depth=RF_MAX_DEPTH,
        max_features=RF_MAX_FEATURES,
        random_state=SEED,
    )
    rf_abl.fit(X_synth_train, y_synth_train)
    rf_test_accs_abl.append(rf_abl.score(X_synth_test, y_synth_test))
    rf_oob_accs_abl.append(rf_abl.oob_score_)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(estimator_counts, rf_test_accs_abl, "o-",  label="Test accuracy",       color="steelblue")
ax.plot(estimator_counts, rf_oob_accs_abl,  "s--", label="OOB accuracy (free)", color="green")
ax.axhline(best_test_acc, color="darkorange", linestyle=":", alpha=0.8,
           label=f"Best single DT ({best_test_acc:.2%})")
ax.set_xlabel("n_estimators")
ax.set_ylabel("Accuracy")
ax.set_title("Random Forest: Number of Trees vs. Accuracy")
ax.legend()
plt.tight_layout()
plt.show()

print(f"Single tree (n=1)    : {rf_test_accs_abl[0]:.2%}")
print(f"Forest (n=100)       : {rf_test_accs_abl[-1]:.2%}")
print(f"OOB closely tracks test accuracy — the gap is {abs(rf_oob_accs_abl[-1] - rf_test_accs_abl[-1]):.4f}")
print("More trees monotonically reduce variance — no overfitting with more trees (unlike deeper single trees).")

### Error Analysis

We examine the Random Forest's misclassifications on the synthetic test set. High-confidence errors (the model is very sure but wrong) suggest the misclassified samples lie in a region dominated by the wrong class — either due to noise, overlapping class distributions, or a genuinely ambiguous feature space.

In [ ]:
# ── Confusion Matrix ──────────────────────────────────────────────────────────
y_pred_rf = rf_synth.predict(X_synth_test)
cm = confusion_matrix(y_synth_test, y_pred_rf)

fig, ax = plt.subplots(figsize=(6, 5))
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=[f"Class {i}" for i in range(N_CLASSES_SYNTH)],
)
disp.plot(ax=ax, colorbar=True, cmap="Blues")
ax.set_title("Random Forest — Confusion Matrix (Synthetic Test Set)")
plt.tight_layout()
plt.show()

# ── Identify most confident errors ────────────────────────────────────────────
misclassified = y_pred_rf != y_synth_test
print(f"Misclassified: {misclassified.sum()} / {len(y_synth_test)}  ({misclassified.mean():.2%} error rate)")
print()
print(classification_report(
    y_synth_test, y_pred_rf,
    target_names=[f"Class {i}" for i in range(N_CLASSES_SYNTH)],
))

# Confident errors: high probability assigned to the wrong class
proba_rf = rf_synth.predict_proba(X_synth_test)
max_confidence = proba_rf.max(axis=1)
error_indices  = np.where(misclassified)[0]

if len(error_indices) > 0:
    conf_of_errors = max_confidence[error_indices]
    sorted_error_pos = np.argsort(conf_of_errors)[::-1]
    n_show = min(5, len(error_indices))
    print(f"Top-{n_show} Most Confident Errors:")
    print(f"  {'Sample':>8} | {'True':>6} | {'Pred':>6} | {'Confidence':>10}")
    print("  " + "-" * 38)
    for pos in sorted_error_pos[:n_show]:
        idx = error_indices[pos]
        print(f"  {idx:>8} | {y_synth_test[idx]:>6} | {y_pred_rf[idx]:>6} | {max_confidence[idx]:>10.2%}")
    print()
    print("High-confidence errors indicate samples in class-boundary ambiguous regions.")
    print("These often correspond to mislabeled training samples or genuinely overlapping classes.")

---
## Part 5 — Summary & Lessons Learned

### Key Takeaways

1. **Decision trees split greedily:** At each node, CART exhaustively searches every (feature, threshold) pair and picks the one minimizing weighted child impurity (Gini or entropy). This greedy strategy is fast but can miss globally optimal splits — post-pruning or depth limits compensate.

2. **Pre-pruning controls the bias-variance tradeoff:** `max_depth`, `min_samples_split`, and `min_samples_leaf` prevent a tree from memorizing training noise. As depth increases, train accuracy reaches 100% while test accuracy plateaus then drops — a textbook overfitting curve.

3. **Bagging decorrelates trees by training on different data:** Each bootstrap sample uses ~63.2% of unique training samples. The unused ~36.8% (out-of-bag samples) provide a *free* unbiased accuracy estimate without a separate validation split.

4. **Feature subsampling decorrelates trees by limiting choices:** Considering only $\sqrt{p}$ features per split prevents the strongest feature from dominating all trees. Combined with bagging, this produces an ensemble where errors are independent — averaging cancels them out.

5. **Impurity-based feature importances are computed bottom-up:** Each internal node's contribution is proportional to the impurity decrease it achieves, weighted by the fraction of samples reaching it. Averaging over all RF trees gives a robust importance ranking for free.

### What's Next

→ **02-04 (Gradient Boosting & AdaBoost)** takes a different ensemble philosophy: instead of parallel bagging, it builds trees *sequentially*, where each new tree corrects the residual errors of the previous ones — achieving lower bias at the cost of more careful regularization.

### Going Further

- **Cost-complexity pruning (post-pruning):** After growing a full tree, prune nodes where the impurity decrease is below a regularization threshold `alpha`. See `sklearn.tree.DecisionTreeClassifier(ccp_alpha=...)` for the implementation.
- **Extremely Randomized Trees (Extra-Trees):** In addition to random feature selection, also randomize the split threshold (pick a random threshold rather than searching the optimal one). Faster training, sometimes better generalization.
- **Isolation Forest:** Uses decision trees for anomaly detection — anomalies are isolated in fewer splits.
- **Breiman (2001)** — "Random Forests" (original paper): https://link.springer.com/article/10.1023/A:1010933404324